In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

if not os.getenv("UPSTAGE_API_KEY"):
    raise RuntimeError(".env 파일에 UPSTAGE_API_KEY 가 없습니다.")
print("UPSTAGE_API_KEY 설정 확인 완료")

UPSTAGE_API_KEY 설정 확인 완료


In [3]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = '../data/콘텐츠분쟁해결_사례.pdf'
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()
print(f"  총 {len(documents)}페이지 로드 완료")

  총 109페이지 로드 완료


In [4]:
print(repr(documents[12].page_content[:300]))

'1. 게임\n7■ 1-가-1) 한국소비자원 조정전 상담사례\n2007_계정 도용 상태에서 발생한 영구이용 제한조치 취소 요구\n【사건개요 】\n신청인은 2007. 8. 24. 피신청인이 제공하는 게임서비스에 접속하였다가 타인이 신청인 계정으로 \n게임을 하고 있는 것을 알게 되어 패스워드를 변경 후 해킹신고를 함. 2007. 8. 27. 해킹신고 처리내역\n을 점검하고자 접속하였더니 사용하지도 않은 불법프로그램 사용으로 인한 계정 영구정지 조치가 \n되어 있어 피신청인에게 수차례 문의하자 불법프로그램 사용으로 인한 제재라는 답변만을 지속적으\n로'


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

pythontext_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,        #  법률 사례는 1200-1800자 권장
    chunk_overlap=300,      #  사례 맥락 보존을 위해 200-400자
    separators=[
        "\n【사건개요 】",     #  법률 문서 섹션 구분자
        "\n【쟁점사항 】",     #  쟁점 부분 구분
        "\n【처리경위 】",     #  처리 과정 구분
        "\n【처리결과 】",     #  결과 부분 구분
        "■", "\n\n", "\n", ".", " ", ""
    ]
)

chunks = pythontext_splitter.split_documents(documents)
print(f"  {len(chunks)}개 청크 생성 완료")
print(f"  평균 청크 길이: {sum(len(chunk.page_content) for chunk in chunks) / len(chunks):.0f}자")


  104개 청크 생성 완료
  평균 청크 길이: 753자


In [6]:
from langchain_upstage import UpstageEmbeddings
from langchain_community.vectorstores import FAISS

pythonembeddings = UpstageEmbeddings(model="solar-embedding-1-large")

vectorstore = FAISS.from_documents(chunks, pythonembeddings)
print(f" FAISS 벡터스토어 생성 완료 ({len(chunks)}개 벡터)")

 FAISS 벡터스토어 생성 완료 (104개 벡터)


In [7]:
pythonretriever = vectorstore.as_retriever(
    search_type="similarity",        #  또는 "mmr" (다양성 고려시)
    search_kwargs={"k": 5}          #  상위 5개 관련 사례 검색
)

In [8]:
from langchain_upstage import ChatUpstage

pythonllm = ChatUpstage(
        model="solar-pro3",
        base_url="https://api.upstage.ai/v1",
        temperature=0.2
    )

In [9]:
def classify_dispute_type(query):
    game_keywords = ["게임", "아이템", "계정", "캐릭터", "레벨", "길드", "온라인게임"]
    elearning_keywords = ["강의", "온라인교육", "이러닝", "수강", "환불", "화상교육"]
    web_keywords = ["웹사이트", "무료체험", "자동결제", "구독", "사이트"]

    query_lower = query.lower()

    if any(keyword in query_lower for keyword in game_keywords):
        return "게임"
    elif any(keyword in query_lower for keyword in elearning_keywords):
        return "이러닝"
    elif any(keyword in query_lower for keyword in web_keywords):
        return "웹콘텐츠"
    else:
        return "기타"

In [10]:
from langchain_core.prompts import PromptTemplate

pythonprompt_template = """
당신은 콘텐츠 분야 전문 법률 자문가입니다. 
아래 분쟁조정 사례들을 바탕으로 정확하고 전문적인 법률 조언을 제공해주세요.

관련 분쟁사례:
{context}

분쟁 유형: {dispute_type}
상담 내용: {question}

답변 가이드라인:
1. 제시된 사례들을 근거로 답변하세요                    # 사례 기반 답변
2. 관련 법령이나 조항이 있다면 명시하세요               # 법적 근거 제시
3. 비슷한 사례의 처리경위와 결과를 참고하여 설명하세요    # 판례 참조
4. 실무적 해결방안을 단계별로 제시하세요               #  실무 가이드
5. 사례에 없는 내용은 "제시된 사례집에서는 확인할 수 없습니다"라고 명시하세요  #  한계 인정

전문 법률 조언:"""

prompt = PromptTemplate.from_template(pythonprompt_template)


In [11]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    """문서들을 문자열로 포맷"""
    return "\n\n".join(doc.page_content for doc in docs)

pythonqa_chain = (
    {
        "context": pythonretriever | format_docs,
        "dispute_type": RunnableLambda(classify_dispute_type),
        "question": RunnablePassthrough(),
    }
    | prompt
    | pythonllm
    | StrOutputParser()
)

In [13]:
pythontest_questions = [
    "온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?",
    "인터넷 강의를 중도 해지하려고 하는데 과도한 위약금을 요구받고 있습니다. 정당한가요?",
    "무료체험 후 자동으로 유료전환되어 요금이 청구되었습니다. 환불 가능한가요?",
    "미성년자가 부모 동의 없이 게임 아이템을 구매했습니다. 환불받을 수 있는 방법이 있나요?",
    "온라인 교육 서비스가 광고와 다르게 제공되어 계약을 해지하고 싶습니다. 가능한가요?"
]

for i, question in enumerate(pythontest_questions, 1):
    answer = pythonqa_chain.invoke(question)

    print(f"테스트 {i}/{len(pythontest_questions)}")
    print(f"분쟁 유형: {classify_dispute_type(question)}")
    print(f"Q: {question}")
    print(f"A: {answer}")
    print("-" * 60)


테스트 1/5
분쟁 유형: 게임
Q: 온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?
A: # 게임 아이템 소멸 관련 법률 자문

## 1. 사례 분석 및 법적 근거

### 관련 법령
- **민법 제250조(도품, 유실물에 대한 특례)**: "도품이나 유실물에 관하여 피해자 또는 유실자는 도난 또는 유실한 날로부터 2년 내에 그 물건의 반환을 청구할 수 있다."
- **민법 제251조(양수인의 선의매수)**: "양수인이 도품 또는 유실물을 경매나 공개시장에서 또는 동 종류의 물건을 판매하는 상인에게서 선의로 매수한 때에는 피해자 또는 유실자는 양수인이 지급한 대가를 변상하고 그 물건의 반환을 청구할 수 있다."
- **전자상거래법 제17조(청약철회)**: "소비자는 계약 체결 후 7일 이내에 청약철회권을 행사할 수 있다."
- **게임산업진흥에 관한 법률 제12조(이용자 보호)**: "게임사업자는 이용자의 개인정보 및 게임 내 자산을 안전하게 관리할 의무가 있다."

### 사례별 분석

#### 사례 1 (2006년 시스템 오류로 소멸된 아이템 복구 요구)
- **쟁점**: 시스템 오류로 인한 아이템 소멸 시 복구 가능 여부
- **처리결과**: 계정 명의자가 아닌 경우 복구 불가
- **법리적 근거**: 피신청인의 약관에 계정 공유 및 현금거래 금지 규정 존재
- **한계**: 계정 명의자 본인인 경우 복구 가능

#### 사례 2 (2009년 시스템 오류로 인한 손실 아이템 복구 요구)
- **쟁점**: 시스템 오류 발생 여부
- **처리결과**: 시스템 오류보다는 통신회선 불안정 또는 PC 오류로 판단
- **법리적 근거**: 게임머니 소실 시 사용 내역 확인 결과 정상 사용 확인
- **한계**: 시스템 오류 입증이 어려운 경우 복구 불가

#### 사례 3 (2006년 프로그램 오류로 소멸된 아이템 복구 요구)
- **쟁점**: 프로그램 오류 여부
- **처리결과**: 프로그램 오